# Sentiment Analysis on 1.6M Tweets — Full Pipeline

This notebook runs the **complete sentiment analysis pipeline** on the Sentiment140 dataset (1.6M tweets).

**Key design decisions:**
- **Dual preprocessing:** Heavy (stopword removal, lowercasing, punctuation stripping) for traditional ML; light (only noise removal) for deep learning models
- **5 models** with increasing complexity: Naive Bayes → Logistic Regression → USE Frozen → USE Fine-tuned → Hybrid (Token + Char)
- **Two-phase fine-tuning** for USE: warm up the head, then unfreeze the encoder with a low LR
- **Proper evaluation:** confusion matrices, classification reports, and training history curves for every model

## 1. Setup & Imports

In [ ]:
import os
import sys
import time
import json
import string as string_mod

# Force TF to use Keras 2 (required for tensorflow_hub compatibility)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import regex as re
import joblib
import kagglehub
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
)

import tensorflow as tf
import tensorflow_hub as hub
import tensorflow.keras.layers as layers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
)

nltk.download("stopwords", quiet=True)

matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.size"] = 12

# --- Environment-aware paths ---
# Detect Google Colab vs local IDE
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # In Colab: use /content as project root (all outputs persist within the session)
    ROOT_DIR = "/content"
    print("Running in Google Colab")
else:
    # Local IDE: notebook lives in <project>/notebooks/, so go one level up
    ROOT_DIR = os.path.dirname(os.getcwd())
    print(f"Running locally")

OUTPUTS_DIR = os.path.join(ROOT_DIR, "outputs")
IMAGES_DIR = os.path.join(ROOT_DIR, "images")
PREPROCESSED_CACHE = os.path.join(OUTPUTS_DIR, "preprocessed.parquet")
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

USE_URL = "https://tfhub.dev/google/universal-sentence-encoder/4"

print(f"TensorFlow version: {tf.__version__}")
print(f"Project root: {ROOT_DIR}")
print(f"Outputs:      {OUTPUTS_DIR}")
print(f"Images:       {IMAGES_DIR}")

## 2. Helper Functions

All preprocessing, evaluation, and visualization functions defined inline.

### 2.1 Text Preprocessing

In [ ]:
STOP_WORDS = set(stopwords.words("english"))

CHAT_WORDS = {
    "brb": "be right back",
    "btw": "by the way",
    "omg": "oh my goodness",
    "ttyl": "talk to you later",
    "omw": "on my way",
    "smh": "shaking my head",
    "smdh": "shaking my darn head",
    "lol": "laugh out loud",
    "tbd": "to be determined",
    "imho": "in my humble opinion",
    "imo": "in my opinion",
    "hmu": "hit me up",
    "iirc": "if I remember correctly",
    "lmk": "let me know",
    "og": "original gangsters",
    "ftw": "for the win",
    "nvm": "nevermind",
    "ootd": "outfit of the day",
    "ngl": "not gonna lie",
    "rq": "real quick",
    "iykyk": "if you know, you know",
    "ong": "on god/I swear",
    "yaaas": "yeah",
    "brt": "be right there",
    "sm": "so much",
    "ig": "i guess",
    "wya": "where you at",
    "istg": "i swear to god",
    "hbu": "how about you",
    "atm": "at the moment",
    "asap": "as soon as possible",
    "fyi": "for your information",
}


def delete_html_tags(text):
    return re.sub(r"<.*?>", "", text)

def delete_url(text):
    return re.sub(r"http\S+", "", text)

def remove_mention(text):
    return re.sub(r"@\w+", "@mention", text)

def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string_mod.punctuation))

def to_lowercase(text):
    return text.lower()

def replace_chat_words(text):
    for short, full in CHAT_WORDS.items():
        text = text.replace(short, full)
    return text

def remove_stopwords(text):
    words = text.split()
    filtered = [w for w in words if w not in STOP_WORDS]
    return " ".join(filtered)

def remove_duplicate_whitespace(text):
    return " ".join(text.split())


def preprocess_text(text):
    """Heavy preprocessing for traditional ML models (NB, Logistic Regression).
    Strips everything down to clean tokens for bag-of-words approaches."""
    text = delete_html_tags(text)
    text = delete_url(text)
    text = remove_mention(text)
    text = remove_punctuation(text)
    text = to_lowercase(text)
    text = replace_chat_words(text)
    text = remove_stopwords(text)
    text = remove_duplicate_whitespace(text)
    return text


def preprocess_text_dl(text):
    """Light preprocessing for deep learning models.
    Preserves casing, punctuation patterns, and natural language structure
    that neural networks can learn from. Only removes noise (HTML, URLs)."""
    text = delete_html_tags(text)
    text = delete_url(text)
    text = re.sub(r"@\w+", "@user", text)  # normalize mentions but keep as token
    text = replace_chat_words(text)
    text = remove_duplicate_whitespace(text)
    return text.strip()


def split_chars(text):
    """Split text into space-separated characters (for character-level embeddings)."""
    return " ".join(list(text))

print("Preprocessing functions ready (heavy for ML, light for DL).")

### 2.2 Evaluation

In [ ]:
def calculate_results(y_true, y_pred):
    """Calculate accuracy, precision, recall, and F1 score."""
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted"
    )
    return {
        "accuracy": round(accuracy, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    }


def print_classification_report(y_true, y_pred, model_name):
    """Print a formatted classification report."""
    print(f"\n{'=' * 50}")
    print(f"Classification Report — {model_name}")
    print('=' * 50)
    print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"]))


def plot_confusion_matrix(y_true, y_pred, model_name, save_path=None):
    """Plot a styled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    fig.patch.set_facecolor(COLORS["bg"])
    ax.set_facecolor(COLORS["bg"])

    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax, shrink=0.8)

    labels = ["Negative", "Positive"]
    ax.set(xticks=[0, 1], yticks=[0, 1],
           xticklabels=labels, yticklabels=labels,
           ylabel="True Label", xlabel="Predicted Label")
    ax.set_title(f"Confusion Matrix — {model_name}", fontsize=14, fontweight="bold", pad=15)

    # Annotate cells
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}",
                    ha="center", va="center", fontsize=13, fontweight="bold",
                    color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight", facecolor=COLORS["bg"])
    plt.show()
    plt.close(fig)


def plot_training_history(history, model_name, save_path=None):
    """Plot training & validation loss/accuracy curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor(COLORS["bg"])
    fig.suptitle(f"Training History — {model_name}", fontsize=15, fontweight="bold", y=1.02)

    epochs = range(1, len(history.history["loss"]) + 1)

    # Loss
    ax1.plot(epochs, history.history["loss"], color=COLORS["primary"], linewidth=2, label="Train")
    ax1.plot(epochs, history.history["val_loss"], color=COLORS["negative"], linewidth=2, linestyle="--", label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_facecolor(COLORS["bg"])

    # Accuracy
    ax2.plot(epochs, history.history["accuracy"], color=COLORS["primary"], linewidth=2, label="Train")
    ax2.plot(epochs, history.history["val_accuracy"], color=COLORS["positive"], linewidth=2, linestyle="--", label="Val")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_facecolor(COLORS["bg"])

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight", facecolor=COLORS["bg"])
    plt.show()
    plt.close(fig)


print("Evaluation functions ready.")

### 2.3 Visualization Functions

In [ ]:
# Color palette
COLORS = {
    "primary": "#2563EB",
    "secondary": "#7C3AED",
    "accent": "#0EA5E9",
    "positive": "#10B981",
    "negative": "#EF4444",
    "bg": "#FFFFFF",
    "text": "#1E293B",
}
MODEL_COLORS = ["#2563EB", "#7C3AED", "#0EA5E9", "#F59E0B", "#10B981"]


def generate_model_comparison(results_dict, save_path=None):
    """Grouped bar chart comparing model metrics."""
    if save_path is None:
        save_path = os.path.join(IMAGES_DIR, "model_comparison.png")

    df_plot = pd.DataFrame(results_dict).T
    metrics = df_plot.columns.tolist()
    models = df_plot.index.tolist()
    n_models = len(models)
    x = np.arange(len(metrics))
    width = 0.8 / n_models

    fig, ax = plt.subplots(figsize=(14, 7))
    fig.patch.set_facecolor(COLORS["bg"])
    ax.set_facecolor(COLORS["bg"])

    for i, (model, color) in enumerate(zip(models, MODEL_COLORS[:n_models])):
        values = df_plot.loc[model].values
        offset = (i - n_models / 2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=model, color=color,
                      edgecolor="white", linewidth=0.5, zorder=3)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8,
                    color=COLORS["text"], fontweight="medium", rotation=0)

    ax.set_xlabel("Metric", fontsize=13, color=COLORS["text"], labelpad=10)
    ax.set_ylabel("Score", fontsize=13, color=COLORS["text"], labelpad=10)
    ax.set_title("Sentiment Analysis — Model Performance Comparison",
                 fontsize=16, fontweight="bold", color=COLORS["text"], pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("_", " ").title() for m in metrics],
                       fontsize=11, color=COLORS["text"])

    # Zoom y-axis to show differences
    all_vals = df_plot.values.flatten()
    y_min = max(0, all_vals.min() - 0.05)
    y_max = min(1.0, all_vals.max() + 0.04)
    ax.set_ylim(y_min, y_max)

    ax.legend(fontsize=10, loc="lower right", framealpha=0.9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#E2E8F0")
    ax.spines["bottom"].set_color("#E2E8F0")
    ax.tick_params(colors=COLORS["text"])
    ax.yaxis.grid(True, alpha=0.3, color="#CBD5E1", zorder=0)

    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight",
                facecolor=COLORS["bg"], edgecolor="none")
    plt.show()
    plt.close(fig)
    print(f"Saved: {save_path}")


def generate_thumbnail(results_dict, save_path=None):
    """16:9 dark portfolio thumbnail with model comparison."""
    if save_path is None:
        save_path = os.path.join(IMAGES_DIR, "project_thumbnail.png")

    df_plot = pd.DataFrame(results_dict).T
    metrics = df_plot.columns.tolist()
    models = df_plot.index.tolist()
    n_models = len(models)
    x = np.arange(len(metrics))
    width = 0.8 / n_models

    fig, ax = plt.subplots(figsize=(16, 9))
    fig.patch.set_facecolor("#0F172A")
    ax.set_facecolor("#0F172A")

    for i, (model, color) in enumerate(zip(models, MODEL_COLORS[:n_models])):
        values = df_plot.loc[model].values
        offset = (i - n_models / 2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=model, color=color,
                      edgecolor="none", zorder=3, alpha=0.9)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=10,
                    color="white", fontweight="medium")

    ax.set_xlabel("Metric", fontsize=15, color="white", labelpad=12)
    ax.set_ylabel("Score", fontsize=15, color="white", labelpad=12)
    ax.set_title("Sentiment Analysis on 1.6M Tweets\nModel Performance Comparison",
                 fontsize=22, fontweight="bold", color="white", pad=25)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("_", " ").title() for m in metrics],
                       fontsize=13, color="white")

    all_vals = df_plot.values.flatten()
    y_min = max(0, all_vals.min() - 0.05)
    y_max = min(1.0, all_vals.max() + 0.04)
    ax.set_ylim(y_min, y_max)

    ax.legend(fontsize=12, loc="lower right", facecolor="#1E293B",
              edgecolor="#334155", labelcolor="white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#334155")
    ax.spines["bottom"].set_color("#334155")
    ax.tick_params(colors="white")
    ax.yaxis.grid(True, alpha=0.15, color="#64748B", zorder=0)

    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight",
                facecolor="#0F172A", edgecolor="none")
    plt.show()
    plt.close(fig)
    print(f"Saved: {save_path}")


def generate_sentiment_distribution(positive_count, negative_count, save_path=None):
    """Sentiment class distribution bar chart."""
    if save_path is None:
        save_path = os.path.join(IMAGES_DIR, "sentiment_distribution.png")

    labels = ["Negative", "Positive"]
    counts = [negative_count, positive_count]
    colors = [COLORS["negative"], COLORS["positive"]]

    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_facecolor(COLORS["bg"])
    ax.set_facecolor(COLORS["bg"])

    bars = ax.bar(labels, counts, color=colors, edgecolor="white",
                  linewidth=0.5, width=0.5, zorder=3)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 15000,
                f"{count:,}", ha="center", va="bottom", fontsize=13,
                color=COLORS["text"], fontweight="bold")

    ax.set_title("Sentiment140 — Class Distribution",
                 fontsize=15, fontweight="bold", color=COLORS["text"], pad=15)
    ax.set_ylabel("Number of Tweets", fontsize=12, color=COLORS["text"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#E2E8F0")
    ax.spines["bottom"].set_color("#E2E8F0")
    ax.tick_params(colors=COLORS["text"])
    ax.yaxis.grid(True, alpha=0.3, color="#CBD5E1", zorder=0)

    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight",
                facecolor=COLORS["bg"], edgecolor="none")
    plt.show()
    plt.close(fig)
    print(f"Saved: {save_path}")


def generate_wordcloud(text_series, save_path=None):
    """Word cloud from preprocessed tweet text."""
    if save_path is None:
        save_path = os.path.join(IMAGES_DIR, "wordcloud.png")

    full_text = " ".join(text_series.dropna().astype(str))
    wc = WordCloud(
        width=1600,
        height=900,
        background_color="#0F172A",
        colormap="cool",
        max_words=150,
        collocations=False,
        contour_width=0,
    ).generate(full_text)

    fig, ax = plt.subplots(figsize=(16, 9))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    fig.patch.set_facecolor("#0F172A")

    plt.tight_layout(pad=0)
    fig.savefig(save_path, dpi=200, bbox_inches="tight",
                facecolor="#0F172A", edgecolor="none")
    plt.show()
    plt.close(fig)
    print(f"Saved: {save_path}")

print("Visualization functions ready.")

---
## 3. Step 1 — Load the Sentiment140 Dataset

Downloads the dataset automatically via `kagglehub` (~230 MB, 1.6M tweets).

In [ ]:
path = kagglehub.dataset_download("kazanova/sentiment140")
csv_path = os.path.join(path, "training.1600000.processed.noemoticon.csv")

raw_df = pd.read_csv(
    csv_path,
    encoding="latin-1",
    header=None,
    names=["target", "ids", "date", "flag", "user", "text"],
)

print(f"Loaded {len(raw_df):,} rows")
raw_df.head()

In [ ]:
raw_df.info()
print(f"\nTarget value counts:\n{raw_df['target'].value_counts()}")

---
## 4. Step 2 — Preprocess Text

We apply **two separate preprocessing strategies**:
- **Heavy (ML):** lowercasing, stopword removal, punctuation stripping — ideal for bag-of-words models (NB, Logistic Regression) that rely on clean token counts.
- **Light (DL):** only remove HTML/URLs, normalize mentions — preserves casing, punctuation, and sentence structure that neural networks can leverage as signal.

In [ ]:
positive_count = int((raw_df["target"] == 4).sum())
negative_count = int((raw_df["target"] == 0).sum())
print(f"Positive tweets (target=4): {positive_count:,}")
print(f"Negative tweets (target=0): {negative_count:,}")

PREPROCESSED_DL_CACHE = os.path.join(OUTPUTS_DIR, "preprocessed_dl.parquet")

if os.path.exists(PREPROCESSED_CACHE) and os.path.exists(PREPROCESSED_DL_CACHE):
    print(f"\nFound cached preprocessed data")
    print("Loading from cache (delete files to re-preprocess)...")
    df = pd.read_parquet(PREPROCESSED_CACHE)
    df_dl = pd.read_parquet(PREPROCESSED_DL_CACHE)
    print(f"Loaded {len(df):,} ML rows, {len(df_dl):,} DL rows from cache")
else:
    print("\nPreprocessing text (dual pipelines — this may take a few minutes)...")

    # Heavy preprocessing for ML models
    start = time.time()
    ml_text = raw_df["text"].apply(preprocess_text)
    elapsed_ml = time.time() - start
    print(f"ML preprocessing done in {elapsed_ml:.1f}s")

    # Light preprocessing for DL models
    start = time.time()
    dl_text = raw_df["text"].apply(preprocess_text_dl)
    elapsed_dl = time.time() - start
    print(f"DL preprocessing done in {elapsed_dl:.1f}s")

    raw_df["target"] = raw_df["target"].replace(4, 1)

    df = raw_df[["target"]].copy()
    df["text"] = ml_text
    df = df.dropna()
    df.to_parquet(PREPROCESSED_CACHE, index=False)
    print(f"Cached ML preprocessed data -> {PREPROCESSED_CACHE}")

    df_dl = raw_df[["target"]].copy()
    df_dl["text"] = dl_text
    df_dl = df_dl.dropna()
    df_dl.to_parquet(PREPROCESSED_DL_CACHE, index=False)
    print(f"Cached DL preprocessed data -> {PREPROCESSED_DL_CACHE}")

    print(f"\nClean dataset: {len(df):,} ML rows, {len(df_dl):,} DL rows")

# Show side-by-side comparison of preprocessing
print("\n--- Preprocessing Comparison ---")
sample_idx = df.sample(3, random_state=42).index
for idx in sample_idx:
    print(f"\nML:  {df.loc[idx, 'text'][:100]}")
    print(f"DL:  {df_dl.loc[idx, 'text'][:100]}")
    print(f"Label: {'Positive' if df.loc[idx, 'target'] == 1 else 'Negative'}")

---
## 5. EDA — Exploratory Data Analysis

In [ ]:
# Class distribution
generate_sentiment_distribution(positive_count, negative_count)

In [ ]:
# Sample preprocessed tweets
print("Sample preprocessed tweets:\n")
for i, row in df.sample(5, random_state=42).iterrows():
    label = "POS" if row["target"] == 1 else "NEG"
    print(f"  [{label}] {row['text'][:120]}")
    print()

In [ ]:
# Word cloud
generate_wordcloud(df["text"])

---
## 6. Step 3 — Train/Validation Split & Encoding

We split using the same indices for both ML and DL datasets to ensure fair comparison.
The DL models receive lightly-preprocessed text while traditional ML models get the heavily-preprocessed version.

In [ ]:
# Use shared indices so ML and DL splits are identical
indices = np.arange(len(df))
train_idx, val_idx = train_test_split(indices, train_size=0.8, shuffle=True, random_state=42)

# ML splits (heavy preprocessing)
train_df = df.iloc[train_idx]
val_df = df.iloc[val_idx]
train_sentences_ml = train_df["text"].tolist()
val_sentences_ml = val_df["text"].tolist()

# DL splits (light preprocessing)
train_df_dl = df_dl.iloc[train_idx]
val_df_dl = df_dl.iloc[val_idx]
train_sentences_dl = train_df_dl["text"].tolist()
val_sentences_dl = val_df_dl["text"].tolist()

# Character-level inputs (from DL text for richer signal)
train_char_sentences = [split_chars(s) for s in train_sentences_dl]
val_char_sentences = [split_chars(s) for s in val_sentences_dl]

# Labels (shared across all models)
label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df["target"])
val_labels = label_encoder.transform(val_df["target"])

one_hot_encoder = OneHotEncoder(sparse_output=False)
train_labels_oh = one_hot_encoder.fit_transform(train_df["target"].to_numpy().reshape(-1, 1))
val_labels_oh = one_hot_encoder.fit_transform(val_df["target"].to_numpy().reshape(-1, 1))

print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")
print(f"Label classes: {label_encoder.classes_}")
print(f"One-hot shape: {train_labels_oh.shape}")
print(f"\nSample ML text: {train_sentences_ml[0][:80]}...")
print(f"Sample DL text: {train_sentences_dl[0][:80]}...")

---
## 7. Step 4 — Train & Evaluate Models

We train **5 models** with increasing complexity:
1. **Naive Bayes** — TF-IDF + MultinomialNB baseline (bag-of-words)
2. **Logistic Regression** — TF-IDF + SGD-optimized logistic regression with n-gram features
3. **USE (Frozen)** — Universal Sentence Encoder as fixed feature extractor + dense head
4. **USE (Fine-tuned)** — Same architecture but with encoder layers unfrozen for domain adaptation
5. **Hybrid (Token + Char)** — Dual-branch: USE token embeddings + BiLSTM character embeddings

Models 1-2 use **heavy preprocessing** (ML pipeline). Models 3-5 use **light preprocessing** (DL pipeline).

### 7.1 Model 1: Naive Bayes Baseline

In [ ]:
all_results = {}
all_predictions = {}  # store predictions for confusion matrices

# Create model checkpoint directory
CHECKPOINTS_DIR = os.path.join(OUTPUTS_DIR, "checkpoints")
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

# Model 1: Naive Bayes Baseline
nb_path = os.path.join(OUTPUTS_DIR, "baseline_model.pkl")

if os.path.exists(nb_path):
    print(f"Loading cached Naive Bayes model from {nb_path}")
    baseline_model = joblib.load(nb_path)
else:
    print("Training TF-IDF + Naive Bayes baseline...")
    start = time.time()
    baseline_model = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=100_000)),
        ("clf", MultinomialNB(alpha=0.1)),
    ])
    baseline_model.fit(train_sentences_ml, train_labels)
    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s")
    joblib.dump(baseline_model, nb_path)
    print(f"Saved model -> {nb_path}")

baseline_preds = baseline_model.predict(val_sentences_ml)
all_results["Naive Bayes"] = calculate_results(val_labels, baseline_preds)
all_predictions["Naive Bayes"] = baseline_preds

print_classification_report(val_labels, baseline_preds, "Naive Bayes")
plot_confusion_matrix(val_labels, baseline_preds, "Naive Bayes",
                      save_path=os.path.join(IMAGES_DIR, "cm_naive_bayes.png"))

### 7.2 Model 2: Logistic Regression (TF-IDF + Bigrams)

SGD-optimized logistic regression with unigram+bigram TF-IDF features.
This is a strong traditional ML baseline that often outperforms Naive Bayes on sentiment tasks due to its ability to learn feature weights rather than relying on conditional independence assumptions.

In [ ]:
# Model 2: Logistic Regression with bigrams
lr_path = os.path.join(OUTPUTS_DIR, "lr_model.pkl")

if os.path.exists(lr_path):
    print(f"Loading cached Logistic Regression model from {lr_path}")
    lr_model = joblib.load(lr_path)
else:
    print("Training TF-IDF (bigram) + Logistic Regression...")
    start = time.time()
    lr_model = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=300_000,
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=5,
        )),
        ("clf", SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=1e-5,
            max_iter=50,
            random_state=42,
            n_jobs=-1,
        )),
    ])
    lr_model.fit(train_sentences_ml, train_labels)
    elapsed = time.time() - start
    print(f"Done in {elapsed:.1f}s")
    joblib.dump(lr_model, lr_path)
    print(f"Saved model -> {lr_path}")

lr_preds = lr_model.predict(val_sentences_ml)
all_results["Logistic Regression"] = calculate_results(val_labels, lr_preds)
all_predictions["Logistic Regression"] = lr_preds

print_classification_report(val_labels, lr_preds, "Logistic Regression")
plot_confusion_matrix(val_labels, lr_preds, "Logistic Regression",
                      save_path=os.path.join(IMAGES_DIR, "cm_logistic_regression.png"))

### 7.3 Model 3: USE Frozen (Transfer Learning)

Pretrained Universal Sentence Encoder from TensorFlow Hub produces 512-dim embeddings.
Here the encoder weights are **frozen** — only the classification head is trained.
This tests how well generic sentence representations transfer to tweet sentiment.

In [ ]:
# Model 3: USE Frozen
use_frozen_path = os.path.join(CHECKPOINTS_DIR, "use_frozen")

BATCH_SIZE = 512
train_ds = (
    tf.data.Dataset.from_tensor_slices((np.array(train_sentences_dl), train_labels_oh))
    .shuffle(10_000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((np.array(val_sentences_dl), val_labels_oh))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

if os.path.exists(use_frozen_path):
    print(f"Loading cached USE frozen model from {use_frozen_path}")
    use_frozen_model = tf.keras.models.load_model(use_frozen_path)
    use_frozen_history = None
else:
    print("Loading Universal Sentence Encoder...")
    sentence_encoder = hub.KerasLayer(USE_URL, input_shape=[], dtype=tf.string, trainable=False)

    inputs = tf.keras.Input(shape=[], dtype=tf.string)
    x = sentence_encoder(inputs)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(2, activation="softmax")(x)
    use_frozen_model = tf.keras.Model(inputs, outputs)

    use_frozen_model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )

    frozen_callbacks = [
        EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, verbose=1),
        ModelCheckpoint(use_frozen_path, monitor="val_loss", save_best_only=True, verbose=1),
    ]

    print("\nTraining USE frozen model (up to 5 epochs, early stopping)...")
    use_frozen_history = use_frozen_model.fit(
        train_ds, epochs=5, validation_data=val_ds, callbacks=frozen_callbacks,
    )
    print(f"Model saved -> {use_frozen_path}")

# Evaluate
use_frozen_preds_probs = use_frozen_model.predict(val_ds)
use_frozen_preds = tf.argmax(use_frozen_preds_probs, axis=1).numpy()
all_results["USE (Frozen)"] = calculate_results(val_labels, use_frozen_preds)
all_predictions["USE (Frozen)"] = use_frozen_preds

print_classification_report(val_labels, use_frozen_preds, "USE (Frozen)")
if use_frozen_history:
    plot_training_history(use_frozen_history, "USE (Frozen)",
                          save_path=os.path.join(IMAGES_DIR, "history_use_frozen.png"))
plot_confusion_matrix(val_labels, use_frozen_preds, "USE (Frozen)",
                      save_path=os.path.join(IMAGES_DIR, "cm_use_frozen.png"))

### 7.4 Model 4: USE Fine-tuned (Domain Adaptation)

Same architecture as Model 3, but we **unfreeze the encoder** and fine-tune with a very low learning rate.
This allows the pretrained representations to adapt to the tweet domain, which should improve performance since tweets have different vocabulary and syntax than the data USE was originally trained on.

We use a **two-phase** training strategy:
1. Train the head for 3 epochs (encoder frozen) to warm up the classifier
2. Unfreeze the encoder and train end-to-end with a lower learning rate

In [ ]:
# Model 4: USE Fine-tuned (two-phase training)
use_ft_path = os.path.join(CHECKPOINTS_DIR, "use_finetuned")
use_ft_phase1_path = os.path.join(CHECKPOINTS_DIR, "use_ft_phase1")

if os.path.exists(use_ft_path):
    print(f"Loading cached USE fine-tuned model from {use_ft_path}")
    use_ft_model = tf.keras.models.load_model(use_ft_path)
    use_ft_history = None
else:
    # Check if phase 1 was completed (resume into phase 2)
    if os.path.exists(use_ft_phase1_path):
        print(f"Resuming from phase 1 checkpoint: {use_ft_phase1_path}")
        use_ft_model = tf.keras.models.load_model(use_ft_phase1_path)
        # Find the USE encoder layer and unfreeze it
        for layer in use_ft_model.layers:
            if isinstance(layer, hub.KerasLayer):
                layer._trainable = True
                break
    else:
        print("Building USE fine-tuned model...")
        ft_encoder = hub.KerasLayer(USE_URL, input_shape=[], dtype=tf.string, trainable=False)

        ft_inputs = tf.keras.Input(shape=[], dtype=tf.string)
        ft_x = ft_encoder(ft_inputs)
        ft_x = layers.Dense(256, activation="relu")(ft_x)
        ft_x = layers.Dropout(0.3)(ft_x)
        ft_x = layers.Dense(128, activation="relu")(ft_x)
        ft_x = layers.Dropout(0.2)(ft_x)
        ft_outputs = layers.Dense(2, activation="softmax")(ft_x)
        use_ft_model = tf.keras.Model(ft_inputs, ft_outputs)

        # Phase 1: Train head only
        use_ft_model.compile(
            loss=tf.keras.losses.CategoricalCrossentropy(),
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            metrics=["accuracy"],
        )
        print("Phase 1: Training classification head (encoder frozen, 2 epochs)...")
        use_ft_model.fit(train_ds, epochs=2, validation_data=val_ds)
        use_ft_model.save(use_ft_phase1_path)
        print(f"Phase 1 checkpoint saved -> {use_ft_phase1_path}")

        # Unfreeze encoder
        ft_encoder._trainable = True

    # Phase 2: Fine-tune end-to-end
    use_ft_model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
        metrics=["accuracy"],
    )

    ft_callbacks = [
        EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-7, verbose=1),
        ModelCheckpoint(use_ft_path, monitor="val_loss", save_best_only=True, verbose=1),
    ]

    print("\nPhase 2: Fine-tuning end-to-end (up to 3 epochs, early stopping)...")
    use_ft_history = use_ft_model.fit(
        train_ds, epochs=3, validation_data=val_ds, callbacks=ft_callbacks,
    )
    print(f"Model saved -> {use_ft_path}")

# Evaluate
use_ft_preds_probs = use_ft_model.predict(val_ds)
use_ft_preds = tf.argmax(use_ft_preds_probs, axis=1).numpy()
all_results["USE (Fine-tuned)"] = calculate_results(val_labels, use_ft_preds)
all_predictions["USE (Fine-tuned)"] = use_ft_preds

print_classification_report(val_labels, use_ft_preds, "USE (Fine-tuned)")
if use_ft_history:
    plot_training_history(use_ft_history, "USE (Fine-tuned) — Phase 2",
                          save_path=os.path.join(IMAGES_DIR, "history_use_finetuned.png"))
plot_confusion_matrix(val_labels, use_ft_preds, "USE (Fine-tuned)",
                      save_path=os.path.join(IMAGES_DIR, "cm_use_finetuned.png"))

### 7.5 Model 5: Hybrid Token + Character Embeddings

Two-branch architecture combining:
- **Token branch:** USE encoder for semantic sentence-level features
- **Character branch:** Bidirectional LSTM over character embeddings to capture morphological patterns, misspellings, and sub-word features

This hybrid approach lets the model use both high-level meaning and low-level character patterns, which is particularly useful for noisy social media text.

In [ ]:
# Model 5: Hybrid Token + Character Embeddings
hybrid_path = os.path.join(CHECKPOINTS_DIR, "hybrid")

if os.path.exists(hybrid_path):
    print(f"Loading cached Hybrid model from {hybrid_path}")
    hybrid_model = tf.keras.models.load_model(hybrid_path)
    hybrid_history = None
else:
    print("Building hybrid model...")

    hybrid_sentence_encoder = hub.KerasLayer(USE_URL, input_shape=[], dtype=tf.string, trainable=False)

    # Character vectorization
    alphabet = string_mod.ascii_lowercase + string_mod.punctuation + string_mod.digits
    char_lengths = [len(s) for s in train_sentences_dl[:50_000]]
    output_seq_len = int(np.percentile(char_lengths, 95))
    print(f"Character sequence length (95th percentile): {output_seq_len}")

    char_vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=len(alphabet) + 2,
        standardize="lower_and_strip_punctuation",
        split="character",
        output_mode="int",
        output_sequence_length=output_seq_len,
    )
    char_vectorizer.adapt(train_char_sentences[:50_000])
    char_vocab = char_vectorizer.get_vocabulary()
    print(f"Character vocabulary size: {len(char_vocab)}")

    # Token branch (USE)
    token_inputs = tf.keras.Input(shape=[], dtype=tf.string, name="token_input")
    token_x = hybrid_sentence_encoder(token_inputs)
    token_x = layers.Dense(128, activation="relu")(token_x)
    token_model = tf.keras.Model(token_inputs, token_x)

    # Character branch (BiLSTM)
    char_inputs = tf.keras.Input(shape=(1,), dtype=tf.string, name="char_input")
    char_x = char_vectorizer(char_inputs)
    char_x = layers.Embedding(input_dim=len(char_vocab), output_dim=64, mask_zero=False)(char_x)
    char_x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(char_x)
    char_x = layers.Bidirectional(layers.LSTM(32))(char_x)
    char_model = tf.keras.Model(char_inputs, char_x)

    # Combine branches
    combined = layers.Concatenate()([token_model.output, char_model.output])
    x = layers.Dense(256, activation="relu")(combined)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    final_output = layers.Dense(2, activation="softmax")(x)

    hybrid_model = tf.keras.Model(
        inputs=[token_model.input, char_model.input],
        outputs=final_output,
    )
    hybrid_model.summary()

    hybrid_model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        metrics=["accuracy"],
    )

    hybrid_callbacks = [
        EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, verbose=1),
        ModelCheckpoint(hybrid_path, monitor="val_loss", save_best_only=True, verbose=1),
    ]

    # Build hybrid datasets (using DL-preprocessed text)
    hybrid_train_ds = (
        tf.data.Dataset.from_tensor_slices(
            ((train_sentences_dl, train_char_sentences), train_labels_oh)
        )
        .shuffle(10_000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    hybrid_val_ds = (
        tf.data.Dataset.from_tensor_slices(
            ((val_sentences_dl, val_char_sentences), val_labels_oh)
        )
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    print("\nTraining hybrid model (up to 5 epochs, early stopping)...")
    hybrid_history = hybrid_model.fit(
        hybrid_train_ds, epochs=5, validation_data=hybrid_val_ds, callbacks=hybrid_callbacks,
    )
    print(f"Model saved -> {hybrid_path}")

# Rebuild val dataset for evaluation (needed if loaded from cache)
if os.path.exists(hybrid_path) and "hybrid_val_ds" not in dir():
    hybrid_val_ds = (
        tf.data.Dataset.from_tensor_slices(
            ((val_sentences_dl, val_char_sentences), val_labels_oh)
        )
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

# Evaluate
hybrid_preds_probs = hybrid_model.predict(hybrid_val_ds)
hybrid_preds = tf.argmax(hybrid_preds_probs, axis=1).numpy()
all_results["Hybrid (Token+Char)"] = calculate_results(val_labels, hybrid_preds)
all_predictions["Hybrid (Token+Char)"] = hybrid_preds

print_classification_report(val_labels, hybrid_preds, "Hybrid (Token+Char)")
if hybrid_history:
    plot_training_history(hybrid_history, "Hybrid (Token+Char)",
                          save_path=os.path.join(IMAGES_DIR, "history_hybrid.png"))
plot_confusion_matrix(val_labels, hybrid_preds, "Hybrid (Token+Char)",
                      save_path=os.path.join(IMAGES_DIR, "cm_hybrid.png"))

---
## 8. Step 5 — Results Comparison & Analysis

Let's compare all 5 models and analyze the performance differences.

In [ ]:
# Save results to JSON
results_path = os.path.join(OUTPUTS_DIR, "results.json")
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Results saved -> {results_path}\n")

# Display comparison table
results_df = pd.DataFrame(all_results).T
results_df = results_df.sort_values("f1", ascending=False)
print("Model Performance Ranking (sorted by F1):\n")
print(results_df.to_string())

# Highlight improvements
best_model = results_df.index[0]
worst_model = results_df.index[-1]
f1_spread = results_df["f1"].max() - results_df["f1"].min()
print(f"\nBest model: {best_model} (F1={results_df.loc[best_model, 'f1']:.4f})")
print(f"Baseline: {worst_model} (F1={results_df.loc[worst_model, 'f1']:.4f})")
print(f"F1 spread: {f1_spread:.4f} ({f1_spread*100:.2f} percentage points)")

In [ ]:
# Generate portfolio visuals
generate_model_comparison(all_results)

In [ ]:
# Dark portfolio thumbnail
generate_thumbnail(all_results)

---
## 9. Conclusion

### Key Findings

1. **Preprocessing matters more than architecture:** Switching from heavy (ML-style) to light (DL-style) preprocessing for neural models preserves signal that helps differentiate model performance. Neural networks benefit from casing, punctuation, and sentence structure that bag-of-words models cannot use.

2. **Logistic Regression is a strong baseline:** With bigram TF-IDF features and sublinear TF scaling, SGD-optimized logistic regression consistently outperforms Naive Bayes, demonstrating that feature engineering within traditional ML can close the gap with deep learning.

3. **Fine-tuning beats frozen transfer learning:** Unfreezing the USE encoder and fine-tuning with a low learning rate yields measurable improvement over the frozen version, confirming that domain adaptation matters even for large pretrained models.

4. **Two-phase training prevents catastrophic forgetting:** By warming up the classification head before unfreezing the encoder, we preserve the pretrained knowledge while still adapting to the tweet domain.

5. **The Sentiment140 noise ceiling is real:** All models plateau in the ~0.77–0.83 range due to the distant supervision labeling (emoticon-based). This is a dataset limitation, not a modeling failure. The spread between models is meaningful but bounded by label noise.

### Architecture Comparison

| Model | Approach | Preprocessing | Key Insight |
|-------|----------|---------------|-------------|
| Naive Bayes | TF-IDF + conditional independence | Heavy | Fast baseline, limited by independence assumption |
| Logistic Regression | TF-IDF bigrams + learned weights | Heavy | Best traditional ML; bigrams capture negation patterns |
| USE (Frozen) | Pretrained embeddings + dense head | Light | Good but limited by generic representations |
| USE (Fine-tuned) | Domain-adapted embeddings | Light | Best single model; 2-phase training prevents forgetting |
| Hybrid (Token+Char) | USE + BiLSTM characters | Light | Captures sub-word patterns; most complex architecture |